# 02 · Três métodos de seleção e votação de features

**Etapa 3.** Três abordagens são executadas **neste mesmo notebook**, somente sobre o **treino** do ciclo (os 20% finais da linha do tempo ficam para validação):

1. **Filtro estatístico** – `SelectKBest(f_regression, k=12)`: relação linear univariada de cada atributo com `cnt`.
2. **Importância por árvores** – `RandomForestRegressor(random_state=42)`: top-12 por `feature_importances_` (captura não linearidades e interações).
3. **Eliminação recursiva** – `RFE(Ridge, n_features_to_select=12)`: seleção multivariada considerando redundância entre atributos.

### Regra de votação (definida antes da comparação – `config.yaml`)
* Cada método vota nas suas **top-12** features.
* É **eleita** a feature com **≥ 2 votos** dos 3 métodos (`min_votos: 2`).
* **Mínimo de 8 features:** se menos de 8 alcançarem 2 votos, completa-se com as melhores posições do **ranking médio ponderado** (pesos iguais, 1/3 cada) entre as não eleitas; empates → mais votos → ordem alfabética.

*Justificativa:* a maioria simples (2 de 3) exige concordância entre métodos de naturezas diferentes (estatístico, não linear e multivariado), reduzindo o viés de cada um; o mínimo de 8 evita um conjunto pobre demais caso os métodos discordem.

In [ ]:
# Setup: garante caminhos consistentes (nbconvert executa com cwd = notebooks/)
import os, sys, warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "config" / "config.yaml").exists())
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image
import mlflow

from src import utils
cfg = utils.carregar_config()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Raiz do projeto:", RAIZ)

In [ ]:
from src import preprocessamento as pp, features as fs

ciclo = utils.carregar_ciclo()
DIR_EV = utils.dir_evidencias(ciclo["cycle_id"])
dados = pd.read_parquet(utils.DIR_PROCESSED / "dataset_ciclo.parquet")
candidatas = pp.lista_features(cfg)

treino, val = pp.separar_treino_validacao(dados, cfg)
X_treino, y_treino = pp.xy(treino, cfg, candidatas)
particao = {
    "treino": [str(treino["dteday"].min().date()), str(treino["dteday"].max().date()), len(treino)],
    "validacao": [str(val["dteday"].min().date()), str(val["dteday"].max().date()), len(val)],
}
print(ciclo["cycle_id"], ciclo["data_version"])
print(json.dumps(particao, indent=2))

## Execução dos três métodos (fit apenas no treino)

In [ ]:
tabela_metodos = fs.executar_metodos(X_treino, y_treino, candidatas, cfg)
tabela_votos, eleitas = fs.votar(tabela_metodos, cfg)

colunas_exibir = ["kbest_score_f", "kbest_rank", "kbest_voto", "rf_importancia", "rf_rank", "rf_voto",
                  "rfe_abs_coef", "rfe_rank", "rfe_voto", "votos", "rank_medio_ponderado", "eleita", "criterio"]
display(tabela_votos[colunas_exibir].round(4))
print(f"\n{len(eleitas)} features eleitas:", eleitas)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
ordem = tabela_votos.sort_values("rank_medio_ponderado").index[::-1]
for ax, (col, titulo, voto) in zip(axes, [("kbest_score_f", "SelectKBest (F)", "kbest_voto"),
                                          ("rf_importancia", "Random Forest (importância)", "rf_voto"),
                                          ("rfe_rank", "RFE-Ridge (posição; menor = melhor)", "rfe_voto")]):
    cores = ["tab:blue" if v else "lightgray" for v in tabela_votos.loc[ordem, voto]]
    ax.barh(ordem, tabela_votos.loc[ordem, col], color=cores); ax.set_title(titulo, fontsize=10)
    ax.tick_params(labelsize=8)
cores = ["tab:green" if e else "lightgray" for e in tabela_votos.loc[ordem, "eleita"]]
axes[3].barh(ordem, tabela_votos.loc[ordem, "votos"], color=cores); axes[3].set_title("Votos (verde = eleita)", fontsize=10)
axes[3].tick_params(labelsize=8)
fig.suptitle(f"Votação de features – {ciclo['cycle_id']} ({ciclo['data_version']})")
fig.tight_layout(); fig.savefig(DIR_EV / "votacao_rankings.png", dpi=110); plt.show()

## Registro da votação (evidências + MLflow)

In [ ]:
assert len(eleitas) >= cfg["votacao"]["min_features"], "Regra do mínimo de features violada"
tabela_votos.to_csv(DIR_EV / "votacao_tabela.csv")

exp_id = utils.configurar_mlflow(cfg)
with mlflow.start_run(run_name=f"{ciclo['cycle_id']}_02_votacao", tags=utils.tags_ciclo(ciclo, "02_votacao")) as run:
    v = cfg["votacao"]
    mlflow.log_params({
        "cycle_id": ciclo["cycle_id"], "data_version": ciclo["data_version"],
        "metodo_votacao": v["metodo"], "k": v["k"], "min_votos": v["min_votos"], "min_features": v["min_features"],
        "pesos_ranking": json.dumps(v["pesos_ranking"]),
        "metodo_1": "SelectKBest(f_regression)", "metodo_2": "RandomForestRegressor.feature_importances_",
        "metodo_3": "RFE(Ridge)", "periodo_treino": f"{particao['treino'][0]}..{particao['treino'][1]}",
        "n_treino": particao["treino"][2], "features_eleitas": ",".join(eleitas),
    })
    mlflow.log_metric("n_features_eleitas", len(eleitas))
    mlflow.log_metrics({f"votos_{f}": int(n) for f, n in tabela_votos["votos"].items()})
    mlflow.log_artifact(str(DIR_EV / "votacao_tabela.csv"), "votacao")
    mlflow.log_artifact(str(DIR_EV / "votacao_rankings.png"), "votacao")
    mlflow.log_dict({"eleitas": eleitas}, "votacao/features_eleitas.json")

resultado = {"cycle_id": ciclo["cycle_id"], "data_version": ciclo["data_version"], "features_eleitas": eleitas,
             "particao": particao, "run_id_votacao": run.info.run_id}
utils.salvar_json(resultado, utils.DIR_PROCESSED / "features_eleitas.json")
utils.salvar_json(resultado, DIR_EV / "features_eleitas.json")
ciclo["run_id_votacao"] = run.info.run_id
ciclo["particao"] = particao
utils.salvar_json(ciclo, utils.CAMINHO_CICLO)
print("Run votação:", run.info.run_id)